In [1]:
import pandas as pd
import numpy as np
import boto3
from io import BytesIO

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
)
BUCKET = "oil-pipeline"

def read_parquet(key):
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))

def save_parquet(df, key):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
    print(f"Saved: {key} ({len(df)} rows)")

In [2]:
production = read_parquet("cleaned/production/data.parquet")
wells = read_parquet("cleaned/wells/wells.parquet")

production["date"] = pd.to_datetime(production["date"])
print(production.shape)
production.head()

(120, 10)


,prod_id,well_id,date,oil_ton,gas_m3,water_m3,energy_kwh,downtime_hours,temperature,pressure
0,1,1,2025-10-01,212.4,55200.0,182.3,7450.0,0.5,88.1,120.4
1,2,1,2025-10-02,213.8,55320.0,181.9,7490.0,0.3,87.8,121.0
2,3,1,2025-10-03,211.9,55040.0,183.2,7430.0,0.7,88.5,119.8
3,4,1,2025-10-04,215.1,55500.0,180.5,7515.0,0.2,87.6,121.5
4,5,1,2025-10-05,214.6,55380.0,182.0,7480.0,0.4,88.0,120.9


In [3]:
mart_daily = production.groupby("date").agg(
    total_oil_ton=("oil_ton", "sum"),
    total_gas_m3=("gas_m3", "sum"),
    total_water_m3=("water_m3", "sum"),
    avg_pressure=("pressure", "mean"),
    avg_temperature=("temperature", "mean"),
    total_downtime_hours=("downtime_hours", "sum"),
).reset_index().sort_values("date")

print(mart_daily)
save_parquet(mart_daily, "marts/mart_production_daily.parquet")

         date  total_oil_ton  total_gas_m3  total_water_m3  avg_pressure  \
0  2025-10-01          717.5      197170.0           631.1       115.425   
1  2025-10-02          717.3      197200.0           630.5       115.475   
2  2025-10-03          719.2      197350.0           630.8       115.500   
3  2025-10-04          722.9      197810.0           627.4       116.000   
4  2025-10-05          721.0      197620.0           630.1       115.725   
5  2025-10-06          718.6      197350.0           632.6       115.475   
6  2025-10-07          714.1      196800.0           635.5       115.075   
7  2025-10-08          716.4      197090.0           631.9       115.375   
8  2025-10-09          721.7      197700.0           628.6       116.000   
9  2025-10-10          718.5      197330.0           631.3       115.500   
10 2025-10-11          714.8      196800.0           634.5       115.125   
11 2025-10-12          718.3      197270.0           631.9       115.525   
12 2025-10-1

In [4]:
mart_wells = production.groupby("well_id").agg(
    avg_oil_ton=("oil_ton", "mean"),
    total_oil_ton=("oil_ton", "sum"),
    avg_pressure=("pressure", "mean"),
    avg_temperature=("temperature", "mean"),
    total_downtime_hours=("downtime_hours", "sum"),
    days_count=("date", "count"),
).reset_index()

mart_wells["downtime_pct"] = (
    mart_wells["total_downtime_hours"] / (mart_wells["days_count"] * 24) * 100
).round(2)

mart_wells = mart_wells.merge(
    wells[["well_id", "name", "region", "operator"]], on="well_id"
).sort_values("avg_oil_ton", ascending=False)

print(mart_wells[["name", "avg_oil_ton", "total_oil_ton", "downtime_pct"]])
save_parquet(mart_wells, "marts/mart_wells_kpi.parquet")

       name  avg_oil_ton  total_oil_ton  downtime_pct
0  Well-101   213.150000         6394.5          2.01
3  Well-305   198.430000         5952.9          1.82
1  Well-102   185.816667         5574.5          3.03
2  Well-203   121.696667         3650.9          8.11
Saved: marts/mart_wells_kpi.parquet (4 rows)


In [5]:
print("Лучшие скважины по добыче:")
print(mart_wells[["name", "avg_oil_ton", "downtime_pct"]].head(3).to_string(index=False))

print("\nХудшие скважины по добыче:")
print(mart_wells[["name", "avg_oil_ton", "downtime_pct"]].tail(3).to_string(index=False))

Лучшие скважины по добыче:
    name  avg_oil_ton  downtime_pct
Well-101   213.150000          2.01
Well-305   198.430000          1.82
Well-102   185.816667          3.03

Худшие скважины по добыче:
    name  avg_oil_ton  downtime_pct
Well-305   198.430000          1.82
Well-102   185.816667          3.03
Well-203   121.696667          8.11


In [6]:
corr_temp = production["oil_ton"].corr(production["temperature"])
corr_pres = production["oil_ton"].corr(production["pressure"])
corr_down = production["oil_ton"].corr(production["downtime_hours"])

print(f"Корреляция oil_ton ↔ temperature:     {corr_temp:.3f}")
print(f"Корреляция oil_ton ↔ pressure:         {corr_pres:.3f}")
print(f"Корреляция oil_ton ↔ downtime_hours:   {corr_down:.3f}")

Корреляция oil_ton ↔ temperature:     0.985
Корреляция oil_ton ↔ pressure:         0.987
Корреляция oil_ton ↔ downtime_hours:   -0.936
